# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 01

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. Аналитик выполнил две строчки кода:
```python
s1 = df['balance']
s2 = df[['balance']]
```
В какой переменной (`s1` или `s2`) получен одномерный столбец (`pd.Series`), а в какой — двумерная таблица (`pd.DataFrame`) из одной колонки? В чем их главное отличие при обращении к методам?

**Ответ:**
* `s1 = df['balance']` возвращает одномерную структуру `pd.Series` (вектор значений).
* `s2 = df[['balance']]` (список в двойных квадратных скобках) возвращает двумерную таблицу `pd.DataFrame` из одной колонки.
К `Series` обращаются как к одномерному вектору, а к `DataFrame` применимы табличные методы (выборка нескольких колонок, отображение в виде таблицы).

---

#### 1.2. Аналитик вызвал `df.info()` для таблицы игровых сессий из 500 строк и увидел запись:
`client_rating: 425 non-null float64`.
Сколько сессий в таблице остались без клиентской оценки (содержат `NaN` / `None`), и какой метод Pandas позволяет подсчитать количество пропусков во всех столбцах?

**Ответ:**
Без оценки остались **75 сессий** (500 - 425 = 75). Для подсчета точного количества пропусков в каждом столбце используется метод `df.isna().sum()` (или `df.isnull().sum()`).

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
zone_analysis["revenue_share_pct"] = (
    zone_analysis["revenue"] / zone_analysis["revenue"].sum() * 100
).round(1)
```

**Ответ:**
Вычисляет процентную долю выручки каждой строки (зоны) в общем объеме выручки всей выборки и округляет полученное значение до 1 десятичного знака. В бизнес-аналитике применяется для расчета структуры выручки.

---

#### 1.4. Аналитик выполнил расчет средней суммы оплат по играм:
```python
report = df.groupby('game_title')['total_paid'].mean()
```
Сколько строк получится в объекте `report` (столько же, сколько всех сессий, или столько, сколько уникальных игр)? Что стало индексом строк в полученном объекте?

**Ответ:**
В объекте `report` строк будет ровно столько, сколько **уникальных игр** в таблице (строки свернулись по уникальным значениям `game_title`). Новым индексом строк стали сами названия игр (`game_title`).

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `members.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_01")
df_audit = pd.read_csv(data_dir / "members.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Средний баланс участников: 1716.67")

Количество строк в таблице: 150
Столбцы датафрейма: ['member_id', 'nickname', 'email', 'city', 'registered_at', 'loyalty_tier', 'balance']
Средний баланс участников: 1716.67

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем полей является точка с запятой (`;`). Так как параметр `sep=';'` не указан, весь файл читается как один столбец, и последующие обращения вызывают `KeyError`.
2. В словаре аргумента `.agg({...})` пропущена запятая между строками `"total_paid": "sum"` и `"duration_hours": "mean"`, что вызывает `SyntaxError`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_01")
df_sessions = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

report = df_sessions.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

report["duration_hours"] = report["duration_hours"].round(2)
report = report.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(report)

,game_title,total_paid,duration_hours
0,Valorant,28369,2.40
1,GTA V,26905,2.29
2,Apex Legends,26390,2.14
3,Dota 2,25958,2.36
4,Counter-Strike 2,23572,2.17
5,Warzone,21734,2.17
6,PUBG,21462,2.15
7,Atomic Heart,20579,2.06
8,Rust,19374,1.90
9,Cyberpunk 2077,18831,2.23


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `members` поле `member_id` объявлено как `PRIMARY KEY` (первичный ключ).
Что произойдет, если система попытается добавить нового клиента с уже существующим номером `member_id = 1` или со значением `NULL`? Почему СУБД вернет ошибку?

**Ответ:**
СУБД заблокирует операцию и вернет ошибку нарушения ограничения целостности (`PRIMARY KEY / UNIQUE / NOT NULL constraint failed`). Значение первичного ключа обязано быть уникальным для каждой записи и не может быть пустым (`NULL`), так как оно однозначно идентифицирует строку.

---

#### 3.2. Администратор киберклуба случайно выполнил запрос `DELETE FROM game_sessions;` вместо удаления одной устаревшей сессии.
Если запрос выполнялся внутри открытой транзакции, какой командой можно отменить удаление до того, как изменения зафиксируются на диске?

**Ответ:**
Командой `ROLLBACK`. Она отменяет все операции текущей открытой транзакции и возвращает данные в исходное состояние (до вызова команды `COMMIT`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT 
    game_title,
    COUNT(*) AS sessions_count,
    SUM(duration_hours) AS total_hours,
    SUM(total_paid) AS total_revenue
FROM game_sessions
GROUP BY game_title
ORDER BY total_revenue DESC;
```

**Ответ:**
Запрос группирует сессии по наименованию игры (`game_title`) и рассчитывает для каждой игры: общее число сессий (`COUNT(*)`), суммарное сыгранное время в часах (`SUM(duration_hours)`) и суммарную выручку клуба (`SUM(total_paid)`). Результат сортируется по убыванию выручки.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_01")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT session_id, member_id, game_title, duration_hours, discount_promo, total_paid
FROM game_sessions
WHERE game_title = 'Counter-Strike 2' AND discount_promo = 1
ORDER BY duration_hours DESC, total_paid DESC
LIMIT 5;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,session_id,member_id,game_title,duration_hours,discount_promo,total_paid
0,20487,1,Counter-Strike 2,5.0,1,1035.0
1,20477,63,Counter-Strike 2,3.0,1,675.0
2,20342,19,Counter-Strike 2,3.0,1,459.0
3,20068,3,Counter-Strike 2,3.0,1,405.0
4,20290,79,Counter-Strike 2,2.5,1,608.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT 
    game_title,
    COUNT(*) AS sessions_count,
    ROUND(SUM(total_paid), 2) AS total_revenue,
    ROUND(AVG(duration_hours), 2) AS avg_duration
FROM game_sessions
GROUP BY game_title
HAVING SUM(total_paid) > 20000
ORDER BY total_revenue DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,game_title,sessions_count,total_revenue,avg_duration
0,Valorant,59,28369.0,2.40
1,GTA V,54,26905.0,2.29
2,Apex Legends,59,26390.0,2.14
3,Dota 2,51,25958.0,2.36
4,Counter-Strike 2,48,23572.0,2.17
5,Warzone,49,21734.0,2.17
6,PUBG,46,21462.0,2.15
7,Atomic Heart,49,20579.0,2.06


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.loyalty_tier,
    COUNT(s.session_id) AS sessions_count,
    ROUND(SUM(s.total_paid), 2) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city, m.loyalty_tier
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,city,loyalty_tier,sessions_count,total_spent
0,20,Thunder_444,Москва,Bronze,7,5317.0
1,19,Paladin_290,Санкт-Петербург,Bronze,9,5202.0
2,31,Neo_156,Санкт-Петербург,Silver,7,4720.0
3,87,Lynx_192,Москва,Platinum,8,4525.0
4,25,Rogue_689,Санкт-Петербург,Gold,7,4245.0
5,45,Thunder_467,Москва,Bronze,8,4112.0
6,42,Ronin_640,Казань,Silver,7,3900.0
7,10,Lynx_628,Москва,Bronze,8,3824.0
8,11,Viper_386,Казань,Bronze,7,3645.0
9,6,Thunder_228,Москва,Silver,8,3612.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT session_id, member_id, game_title, duration_hours, total_paid
FROM game_sessions
WHERE total_paid > (SELECT AVG(total_paid) FROM game_sessions)
ORDER BY total_paid DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,session_id,member_id,game_title,duration_hours,total_paid
0,20437,93,GTA V,5.0,2100.0
1,20149,20,PUBG,5.0,1900.0
2,20009,79,PUBG,5.0,1800.0
3,20040,60,Dota 2,6.0,1620.0
4,20214,31,Cyberpunk 2077,4.0,1600.0
5,20429,6,Apex Legends,6.0,1458.0
6,20376,87,Valorant,6.0,1380.0
7,20146,19,Dota 2,6.0,1350.0
8,20450,45,Apex Legends,5.0,1350.0
9,20310,91,Valorant,3.0,1260.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Выручка клуба по играм

**Найденные ошибки:**
1. Агрегатная функция в `WHERE`: условие `WHERE SUM(total_paid) > 15000` недопустимо, фильтрация агрегированных значений должна выполняться в `HAVING`.

In [9]:
query_51_fixed = '''
SELECT 
    game_title,
    COUNT(*) AS cnt,
    SUM(total_paid) AS total_revenue
FROM game_sessions
GROUP BY game_title
HAVING SUM(total_paid) > 15000
ORDER BY total_revenue DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,game_title,cnt,total_revenue
0,Valorant,59,28369.0
1,GTA V,54,26905.0
2,Apex Legends,59,26390.0
3,Dota 2,51,25958.0
4,Counter-Strike 2,48,23572.0
5,Warzone,49,21734.0
6,PUBG,46,21462.0
7,Atomic Heart,49,20579.0
8,Rust,46,19374.0
9,Cyberpunk 2077,39,18831.0


#### 5.2. Траты клиентов на игровое время

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает идентификатор клиента с номером компьютера вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: фильтрация по `SUM(s.total_paid) > 1000` должна выполняться в секции `HAVING`, а не в `WHERE`.

In [10]:
query_52_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    SUM(s.total_paid) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname
HAVING SUM(s.total_paid) > 1000
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,member_id,nickname,total_spent
0,20,Thunder_444,5317.0
1,19,Paladin_290,5202.0
2,31,Neo_156,4720.0
3,87,Lynx_192,4525.0
4,25,Rogue_689,4245.0
5,45,Thunder_467,4112.0
6,42,Ronin_640,3900.0
7,10,Lynx_628,3824.0
8,11,Viper_386,3645.0
9,6,Thunder_228,3612.0


#### 5.3. Сравнение клиентов со средней длительностью сессий клуба

**Найденные ошибки:**
1. Порядок команд: блок `WHERE` написан после `GROUP BY`, фильтрация агрегатов должна выполняться через `HAVING`.
2. Использование псевдонима: в фильтрации указан alias `avg_member_hours` вместо агрегатной функции `AVG(s.duration_hours)`.
3. Многострочный подзапрос: наличие `GROUP BY game_title` в подзапросе возвращает вектор значений, несовместимый со скалярным оператором `>`.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    ROUND(AVG(s.duration_hours), 2) AS avg_member_hours
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING AVG(s.duration_hours) > (
    SELECT AVG(duration_hours) 
    FROM game_sessions
)
ORDER BY avg_member_hours DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,city,avg_member_hours
0,73,Shade_347,Казань,4.50
1,51,Falcon_791,Москва,3.75
2,117,Ronin_970,Москва,3.70
3,35,Turbo_145,Москва,3.50
4,81,Grizzly_566,Екатеринбург,3.50
5,60,Ranger_746,Москва,3.33
6,66,Cobra_751,Москва,3.20
7,20,Thunder_444,Москва,3.14
8,79,Rogue_607,Новосибирск,3.13
9,88,Thunder_707,Москва,3.13
